# Smart Travel Planner — ML Training Notebook

This notebook covers:
- Data preprocessing & EDA
- TF-IDF content-based features
- KMeans user clustering
- RandomForest budget prediction
- Model evaluation & persistence (joblib)

In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

DATA_PATH = ROOT / "datasets" / "attractions.csv"
MODEL_OUT = ROOT / "models" / "trained_models.pkl"

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Load & Preprocess Data

In [ ]:
df = pd.read_csv(DATA_PATH)
df["category"] = df["category"].str.lower().str.strip()
df["text_features"] = df["category"] + " " + df["tags"] + " " + df["description"]
print(f"Records: {len(df)}")
df.head()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df["city"].value_counts().plot(kind="bar", ax=axes[0], color="#1a5f7a")
axes[0].set_title("Attractions per City")
df["category"].value_counts().plot(kind="barh", ax=axes[1], color="#159895")
axes[1].set_title("Category Distribution")
axes[2].hist(df["rating"], bins=15, color="#20c997", edgecolor="white")
axes[2].set_title("Rating Distribution")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for numeric features
numeric_cols = ["rating", "estimated_cost", "duration_hours", "popularity"]
corr = df[numeric_cols].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap="YlGnBu", fmt=".2f")
plt.title("Feature Correlation Heatmap")
plt.show()

## 3. TF-IDF & Cosine Similarity (Content-Based)

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", max_features=500)
tfidf_matrix = vectorizer.fit_transform(df["text_features"])

user_profile = "nature adventure photography museum"
profile_vec = vectorizer.transform([user_profile])
similarities = cosine_similarity(profile_vec, tfidf_matrix).flatten()
df["cosine_sim"] = similarities
top = df.nlargest(10, "cosine_sim")[["name", "city", "category", "rating", "cosine_sim"]]
top

## 4. KMeans User Clustering

In [ ]:
from utils.clustering import UserClusterer

kmeans, scaler, label_map = UserClusterer.train_synthetic_kmeans(n_samples=600)
print("Cluster labels:", label_map)

In [ ]:
# Visualize clusters (2D PCA projection of synthetic users)
from sklearn.decomposition import PCA

rng = np.random.default_rng(42)
INTEREST_KEYS = ["nature", "cafe", "museum", "food", "nightlife", "photography", "adventure"]
STYLE_MAP = {"budget": 0, "normal": 1, "luxury": 2}
X = []
for _ in range(300):
    interests = rng.choice(INTEREST_KEYS, size=rng.integers(2, 5), replace=False)
    style = rng.choice(["budget", "normal", "luxury"])
    days = int(rng.integers(2, 7))
    budget = float(rng.integers(3_000_000, 30_000_000))
    vec = [1.0 if k in interests else 0.0 for k in INTEREST_KEYS]
    vec += [budget, days, float(STYLE_MAP[style]), budget / days]
    X.append(vec)
X = np.array(X)
X_scaled = scaler.transform(X)
labels = kmeans.predict(X_scaled)
pca = PCA(n_components=2)
xy = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
for cid in range(4):
    mask = labels == cid
    plt.scatter(xy[mask, 0], xy[mask, 1], label=label_map[cid], alpha=0.7, s=40)
plt.legend()
plt.title("KMeans Traveler Clusters (PCA)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

## 5. Random Forest Budget Prediction

In [ ]:
from utils.predictor import BudgetPredictor

budget_model, metrics = BudgetPredictor.train_from_attractions(df, n_samples=1000)
print("Evaluation metrics:", metrics)

In [ ]:
# Feature importance from Random Forest
feature_names = [f"int_{k}" for k in INTEREST_KEYS] + ["num_days", "style", "num_places", "stated_budget"]
importances = budget_model.feature_importances_
idx = np.argsort(importances)[::-1][:10]
plt.figure(figsize=(8, 4))
plt.barh([feature_names[i] for i in idx[::-1]], importances[idx[::-1]], color="#0d6efd")
plt.title("Top Feature Importances — Budget Model")
plt.tight_layout()
plt.show()

## 6. Save Models (joblib)

In [ ]:
bundle = {
    "kmeans": kmeans,
    "scaler": scaler,
    "label_map": label_map,
    "budget_model": budget_model,
    "metrics": metrics,
}
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(bundle, MODEL_OUT)
print(f"Saved models to {MODEL_OUT}")

## Summary

- **TF-IDF** encodes attraction text for content-based matching
- **KMeans** assigns traveler personas: Backpacker, Explorer, Luxury Traveler, Foodie
- **RandomForest** predicts total trip cost with MAE/R² reported above
- Models are loaded by `utils/model_loader.py` in the Flask app